<a href="https://colab.research.google.com/github/Computercoder125/CSC121Projects/blob/main/heart_disease_analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Sean Gor
Heart Disease Analysis
10/20/26

1. **How do the feature types in the UCI Heart Disease dataset (combining continuous physiological telemetry like chol and thalch with categorical/ordinal indicators like cp and thal) differ from previous datasets, and why is proper scaling critical for Support Vector Machines compared to Decision Trees?**

This dataset mixes continuous measurements (age, trestbps, chol, thalch, oldpeak), nominal categories (sex, dataset, cp, restecg, thal) and binary or ordinal indicators (fbs, exang, slope, ca), while many earlier datasets are mostly numeric and already clean. The features are on very different scales (chol reaches 603, while oldpeak is mostly 0 to 3), some columns that look numeric are really categories (ca, fbs, exang), and the data combines four hospitals with very different disease rates. This required a ColumnTransformer that scales the numeric columns and one-hot encodes the categorical ones, turning 14 features into 32 columns. Scaling is critical for the SVC because the RBF kernel depends on distances between patients. Without scaling, a large-range feature such as cholesterol would dominate the distance and swamp a smaller-range but clinically important feature such as oldpeak. I used StandardScaler instead of MinMaxScaler because chol has extreme outliers that min-max scaling would squash into a narrow range. A Decision Tree does not need scaling because it only asks threshold questions about one feature at a time, and rescaling does not change the ordering of values or the splits that are chosen.

2. **Explain how hidden missing values (such as ? symbols) impact raw medical datasets, and justify your choice of imputation strategies for skewed numerical vitals versus categorical diagnostic fields.**

Hidden missing values are dangerous because nothing flags them as missing. A ? makes a numeric column load as text, which breaks scaling and most models. An impossible value such as a cholesterol of 0 looks like a real reading, so it distorts the mean, the standard deviation and the scaling, and the model learns from a measurement that never happened. In medical data, missingness can also carry information (a test may be skipped for low-risk patients), so mishandling it can bias results. My file had no ? strings, because it uses blank cells, but 172 cholesterol readings and 1 blood pressure reading were recorded as 0. Converting those to NaN raised the missing rate for chol from 3% to 22%, and the most missing columns were ca (66%), thal (53%) and slope (34%). I filled the numeric vitals with the median, because they are skewed (chol skewness 1.31, oldpeak 1.04) and the mean gets pulled toward the long tail (for example, oldpeak has a mean of 0.9 but a median of 0.5). I filled the categorical fields with the most frequent value, because a mean or median has no meaning for categories and the mode does not create an invalid category. Both imputers are inside the pipeline, so they are fit only on the training data, which prevents data leakage. A limitation is that filling more than half of ca and thal with one value is a weak assumption.

3. **Contrast how a Support Vector Classifier (using an RBF kernel) constructs decision boundaries versus how a Decision Tree recursively partitions feature space. What are the interpretability tradeoffs between the two models?**

The SVC finds the boundary that separates the classes with the widest margin. The RBF kernel implicitly maps the data into a high-dimensional space, where a flat boundary becomes a smooth, curved, non-linear boundary in the original space. The boundary is defined by support vectors, and because it uses all features at once it captures interactions between them automatically. A Decision Tree instead recursively splits the space with axis-aligned cuts, one feature and one threshold per node, chosen to reduce Gini impurity. This produces rectangular regions with a blocky boundary, and an unrestricted tree keeps splitting until the training data is perfect, which happened here (100% training accuracy, depth 17, 138 leaves, but only 72.8% test accuracy). On interpretability, the tree is transparent: every prediction follows a readable chain of if-then rules that a clinician can check. The SVC is a black box, since the RBF kernel has no per-feature coefficients and single predictions are hard to explain. The SVC was more accurate (85.9% accuracy, AUC 0.917, 6 missed disease cases) than the tree (72.8%, AUC 0.727, 27 missed cases), so the tradeoff is accuracy against explainability.

4. **Discuss how tuning hyperparameters such as regularization parameter C and kernel coefficient gamma in an SVC impacts the trade-off between bias, variance, and model generalization on unseen medical test data.**

C controls how heavily misclassified training points are penalized. A small C gives a wide margin and a simple boundary with higher bias and lower variance (underfitting), while a large C forces the model to classify almost every training point correctly, giving lower bias and higher variance (overfitting to noise and outliers). gamma controls how far a single training point’s influence reaches. A small gamma gives a smooth, near-linear boundary (high bias, low variance), while a large gamma makes each point influence only its close neighbors, so the boundary wraps around individual patients and the model can memorize the training set (low bias, high variance). The two interact, with large C plus large gamma the most overfit and small C plus small gamma the most underfit, so they should be tuned together. I ran GridSearchCV with C in {0.1, 1, 10, 100} and gamma in {'scale', 0.001, 0.01, 0.1, 1}, using 5-fold stratified cross-validation scored by ROC-AUC on the training data only. The best combination was C = 1 and gamma = 'scale' (cross-validation AUC 0.899), which are the defaults, so the test results were unchanged (85.9% accuracy, AUC 0.917). Many other settings scored within about 0.003, less than the fold-to-fold variation of about 0.02, and only C = 0.1 was noticeably lower (0.888), consistent with underfitting. Cross-validation estimates generalization without using the test set, and the wide plateau of similar scores suggests the model is stable, which is desirable for clinical use.

**5. Inspecting your Decision Tree feature importances or model coefficients, which clinical indicators (e.g., maximum heart rate, chest pain type, ST depression) exhibit the strongest predictive power for heart disease, and what preventative screening insights do these results provide for cardiologists?**

In the fully grown tree, the top features were cp_asymptomatic (0.241), chol (0.124), age (0.111), thalch (0.107), oldpeak (0.085) and trestbps (0.068). In the depth-3 tree, cp_asymptomatic dominated (0.604), followed by age (0.122), oldpeak (0.072), exang_True (0.051) and sex_Male (0.047). The raw data supports this: 79.0% of asymptomatic patients have disease (versus 13.8% to 43.5% for other chest pain types), 83.7% of patients with exercise-induced angina have disease (versus 36.4% without), mean ST depression is 1.26 with disease versus 0.42 in healthy patients, mean maximum heart rate is 128 versus 149 bpm, and 63.2% of males have disease versus 25.8% of females. For cardiologists, this suggests that the absence of chest pain is not reassuring (consistent with silent ischemia), so screening should rely on objective tests and risk factors and not on symptoms alone. Stress test findings (exercise-induced angina, ST depression and a low maximum heart rate) should be weighted heavily when deciding who needs further imaging, and older male patients are a good target for screening. There are caveats: these are associations and not proof of causation, so the model should support clinical judgment and not replace it. dataset_Switzerland appeared as a predictor, which means the tree partly learns which hospital a patient came from, so dropping that column would be an improvement. Impurity-based importances can also favor continuous features and vary between splits, chol is unreliable because 22% of its values were imputed, and ca and thal were mostly missing.